<a href="https://colab.research.google.com/github/yassersherif2005-web/arabic-sentiment-classifier/blob/main/arabic_sentiment_classifier_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

from datasets import load_dataset
dataset = load_dataset("arbml/Arabic_Sentiment_Twitter_Corpus")
print(dataset)

README.md:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.13MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  783kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11751 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 47000
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 11751
    })
})


In [2]:
import pandas as pd

df = pd.DataFrame(dataset["train"])
print(df.head(10))
print(df["label"].value_counts())

                                               tweet  label
0  اعترف ان بتس كانو شوي شوي يجيبو راسي لكن اليوم...      0
1  توقعت اذا جات داريا بشوفهم كاملين بس لي للحين ...      0
2  #الاهلي_الهلال اكتب توقعك لنتيجة لقاء الهلال و...      0
3  نعمة المضادات الحيوية . تضع قطرة💧مضاد بنسلين ع...      0
4                             الدودو جايه تكمل علي 💔      0
5  أحببته حتى أقنعنى إن ما فات من العمر كان إنتظا...      0
6  يبدو ان دجلة اعتادت على التهام اجساد ابنائها م...      0
7                                       جالس أذاكر 😣      0
8  يكفي استخفاف بالعقل العربي هل حقا الأمن والمخا...      0
9                         ياليلل ترا اكلج كثير مشي 😭      0
label
1    23879
0    23121
Name: count, dtype: int64


In [3]:
import re

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^ء-ي\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_tweet"] = df["tweet"].apply(clean_text)
print(df[["tweet", "clean_tweet"]].head(5))


                                               tweet  \
0  اعترف ان بتس كانو شوي شوي يجيبو راسي لكن اليوم...   
1  توقعت اذا جات داريا بشوفهم كاملين بس لي للحين ...   
2  #الاهلي_الهلال اكتب توقعك لنتيجة لقاء الهلال و...   
3  نعمة المضادات الحيوية . تضع قطرة💧مضاد بنسلين ع...   
4                             الدودو جايه تكمل علي 💔   

                                         clean_tweet  
0  اعترف ان بتس كانو شوي شوي يجيبو راسي لكن اليوم...  
1  توقعت اذا جات داريا بشوفهم كاملين بس لي للحين ...  
2  الاهلي الهلال اكتب توقعك لنتيجة لقاء الهلال وا...  
3  نعمة المضادات الحيوية تضع قطرة مضاد بنسلين على...  
4                               الدودو جايه تكمل علي  


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

test_df = pd.DataFrame(dataset["test"])
test_df["clean_tweet"] = test_df["tweet"].apply(clean_text)

vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(df["clean_tweet"])
X_test = vectorizer.transform(test_df["clean_tweet"])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, df["label"])

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(test_df["label"], preds))

Accuracy: 0.758318440983746


In [9]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    return "إيجابي 😊" if pred == 1 else "سلبي 😠"

print(predict_sentiment("ماتش الكوره دا كان جامد  فشخ"))
print(predict_sentiment("أوسخ تجربة في حياتي، خدمة وحشة"))

إيجابي 😊
سلبي 😠


In [10]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(df["clean_tweet"])
X_test = vectorizer.transform(test_df["clean_tweet"])

model = LogisticRegression(max_iter=1000)
model.fit(X_train, df["label"])

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(test_df["label"], preds))

Accuracy: 0.764445579099651
